Loading Pytorch

In [1]:
!pip install torch

Loading libraries

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Load + report-level aggregation (identical to the notebook)

In [11]:
df = pd.read_csv("adverse_events.csv", parse_dates=["receive_date"])

def to_years(row):
    if pd.isna(row["patient_age"]):
        return np.nan
    unit = str(row["patient_age_unit"]).lower()
    if unit.startswith("year"):
        return row["patient_age"]
    if unit.startswith("month"):
        return row["patient_age"] / 12
    if unit.startswith("week"):
        return row["patient_age"] / 52
    if unit.startswith("day"):
        return row["patient_age"] / 365
    return row["patient_age"]

df["age_years"] = df.apply(to_years, axis=1)
df["report_year"] = df["receive_date"].dt.year
df["reactions_per_report"] = df.groupby("safetyreportid")["reaction"].transform("count")
df["serious"] = df["serious"].astype(bool)

report_level = (
    df.groupby("safetyreportid")
    .agg(
        generic_name=("generic_name", "first"),
        country=("country", "first"),
        patient_sex=("patient_sex", "first"),
        age_years=("age_years", "first"),
        patient_weight_kg=("patient_weight_kg", "first"),
        report_year=("report_year", "first"),
        reactions_per_report=("reactions_per_report", "first"),
        n_unique_reactions=("reaction", "nunique"),
        serious=("serious", "first"),
    )
    .reset_index(drop=True)
)

num_features = ["age_years", "patient_weight_kg", "reactions_per_report",
                 "n_unique_reactions", "report_year"]
cat_features = ["generic_name", "country", "patient_sex"]

X = report_level[num_features + cat_features]
y = report_level["serious"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

 2. Preprocess with sklearn (fit on train only), output dense arrays

In [12]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_features),
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
# OneHotEncoder output can be sparse — densify for torch tensors
if hasattr(X_train_proc, "toarray"):
    X_train_proc = X_train_proc.toarray()
    X_test_proc = X_test_proc.toarray()

n_features = X_train_proc.shape[1]


3. Dataset / DataLoader


In [13]:
class ReportDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(ReportDataset(X_train_proc, y_train), batch_size=256, shuffle=True)
test_loader = DataLoader(ReportDataset(X_test_proc, y_test), batch_size=512, shuffle=False)

 4. Model — small feedforward net


In [14]:
class SeriousnessClassifier(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),  # raw logit; use BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x)

model = SeriousnessClassifier(n_features).to(device)

# class_weight="balanced" equivalent: weight the positive class
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

5. Train

In [15]:
EPOCHS = 20
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{EPOCHS} - train loss: {total_loss / len(train_loader.dataset):.4f}")


Epoch  1/20 - train loss: 0.7245
Epoch  5/20 - train loss: 0.6774
Epoch 10/20 - train loss: 0.6726
Epoch 15/20 - train loss: 0.6688
Epoch 20/20 - train loss: 0.6618


6. Evaluate

In [16]:
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        probs = torch.sigmoid(model(xb)).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(yb.numpy())

y_proba = np.concatenate(all_probs).ravel()
y_true = np.concatenate(all_labels).ravel()
y_pred = (y_proba >= 0.5).astype(int)

auc = roc_auc_score(y_true, y_proba)
print(f"\nROC-AUC: {auc:.3f}")
print(classification_report(y_true, y_pred, target_names=["Not serious", "Serious"]))


ROC-AUC: 0.855
              precision    recall  f1-score   support

 Not serious       0.87      0.86      0.86      7734
     Serious       0.66      0.67      0.66      3100

    accuracy                           0.81     10834
   macro avg       0.76      0.76      0.76     10834
weighted avg       0.81      0.81      0.81     10834

